# Task 3: Generate a blog-style article for each product category

**Input:** `reviews_clustered.csv` from the clustering notebook (every review with its meta-category)
**Goal:** for each category, generate a short article with the top 3 products and how they differ, their top complaints, and the worst product to avoid.

## The approach: facts first, then writing

Small language models (1B-8B parameters) write fluent text, but they invent numbers and details if we just hand them thousands of reviews. So we split the job in two:

1. **Pandas computes the facts:** ratings, share of negative reviews, ranking, complaint themes, example complaints. These are exact and checkable.
2. **The language model only writes the article** from a compact fact sheet, and is told to use nothing else.

We then **compare three prompt variants** and score the articles automatically (are the numbers real? are all products and complaints mentioned? is the length right?), as the brief recommends.

**Steps**
1. Load the data and attach verified product names
2. Compute per-product statistics and complaint themes
3. Build a fact sheet per category
4. Load the language model
5. Compare three prompts with automatic checks
6. Generate and save the final articles

In [1]:
!pip install -q transformers accelerate

In [ ]:
import re
import json
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from google.colab import drive
drive.mount("/content/drive")

SEED = 42
REVIEWS_PATH = "/content/drive/MyDrive/Ironhack_Shared/Workshops/NLP_Customer_Reviews/data/reviews_clustered.csv"   # output of the clustering notebook; adjust the path if needed

MIN_REVIEWS_RANK = 30    # a product needs this many reviews to be ranked (avoids "best" with 3 reviews)
MIN_NEG = 15             # fewer negative reviews than this: widen the complaint pool to 1-3 stars
MIN_CATEGORY_REVIEWS = 100  # categories with fewer reviews get no article (too little evidence)
MAX_NEW_TOKENS = 900       # 650 cut one article off mid-sentence
RANK_BY = "popularity"   # "popularity": the 3 most-reviewed models | "rating": the 3 best-rated (smoothed) models
SMOOTHING = 50           # shrinks the rating of products with few reviews toward the category average
STOP = list(ENGLISH_STOP_WORDS | {"br"})
GENERIC = {"amazon", "kindle", "kindles", "fire", "tablet", "tablets", "good", "great", "nice", "love", "loves",
           "bought", "work", "works", "recommend", "like", "really", "use", "used", "product", "got", "buy",
           "new", "day", "gift", "just", "better", "best", "easy", "price", "well", "old", "year", "years"}

Mounted at /content/drive


## 1. Load the data and attach verified product names

The `name` column in this dataset is unreliable: the same product ID often carries several unrelated names. In the clustering notebook we saw that the categories and review text describe a product better than its name.

So I built a **verified name per product ID**, and listings of the same model (for example the 16 GB and 32 GB Fire HD 8) share one name so their reviews are pooled by comparing the raw names with the categories and the words reviewers use. The evidence is in the comments. The names are my best reading of the data, so check them in the table below before publishing anything. Products not in the dictionary, and those marked "model unclear", are never ranked: an article should not recommend a product we cannot name.

In [3]:
PRODUCT_NAMES = {
    # asin: product name. Different listings of the SAME model share one name, so their reviews are pooled.
    # (evidence: most common words in the reviews and the categories)
    "B018Y229OU": 'Fire Tablet 7"',                 # 8 GB listing; tablet reviews
    "B018Y225IA": 'Fire Tablet 7"',                 # 16 GB listing (name: Kindle Fire 16 GB 7" tablet)
    "B01AHB9CN2": "Fire HD 8 Tablet",               # 16 GB listing
    "B01AHB9CYG": "Fire HD 8 Tablet",               # 32 GB listing
    "B018SZT3BK": "Fire HD 8 Tablet",               # reviews say "hd8", "6th"
    "B018Y23MNM": "Fire Kids Edition Tablet",       # reviews: toddlers, parental controls
    "B0189XYY0Q": "Fire HD 10 Tablet",              # reviews say "hd10", "10th"
    "B00TSUGXKE": "Fire Tablet (model unclear)",    # name says Echo, but categories and reviews are tablet-like
    "B00L9EPT8O,B01E6AO69U": "Amazon Echo / Echo Dot",  # reviews: echo, dot, alexa (names say Fire TV for half the rows)
    "B01BH83OOM": "Amazon Tap",                     # reviews: tap, portability; categories: Amazon Tap
    "B00U3FPN4U": "Amazon Fire TV",                 # reviews: firetv, roku, 4k, remote
    "B00OQVZDJM": "Kindle Paperwhite",              # reviews: backlight, sunlight; name: Paperwhite
    "B00IOY8XWQ": "Kindle Voyage",                  # reviews: voyage, page-press buttons
    "B00VINDBJK": "Kindle Oasis",                   # reviews: oasis, leather cover
    "B00ZV9PXP2": "Kindle E-reader (model unclear)",# name says Fire Tablet, but reviews and categories are e-reader
    "B01J2G4VBG": "Amazon 5W USB Charger",          # reviews: charger, 5w, adapter
    "B00QFQRELG": "Amazon 9W PowerFast Charger",    # reviews: powerfast, 9w
    # --- smaller products. The raw name is clear but CONTRADICTS the categories and reviews, so we follow those ---
    "B00IOYAM4I": "Kindle Voyage",                  # 51 reviews. Raw name says Fire tablet; categories are E-readers, reviews say "voyage"
    "B01BFIBRIE": "Kindle Paperwhite",              # 30 reviews. Raw name says Fire tablet; reviews say "paperwhite", "books"
    "B00QJDU3KY": "Kindle Paperwhite",              # 15 reviews. Reviews say "paperwhite", "reading" (lower confidence)
    "B01AHB9C1E": "Fire HD 8 Tablet",               # 14 reviews. Categories: Fire Tablets; some rows name "Fire HD 8 ... 32 GB"
    "B018Y22BI4": "Fire Tablet (model unclear)",    # 10 reviews. Name says Echo; categories say Fire Tablets
    # --- tiny products, named by type. These are never ranked (they start with "Other product") ---
    "B00QL1ZN3G": "Other product: charger or cable",
    "B006GWO5WK": "Other product: charger or cable",
    "B006GWO5NE": "Other product: charger or cable",
    "B01J4ORNHU": "Other product: charger or cable",
    "B00LW9XOJM": "Other product: case or cover",
    "B002Y27P6Y": "Other product: case or cover",
    "B00UH4D8G2": "Other product: Fire TV box",
}

reviews = pd.read_csv(REVIEWS_PATH)
reviews["product"] = reviews["asin"].map(PRODUCT_NAMES).fillna("Other product " + reviews["asin"].astype(str).str[:10])
# A product model belongs to ONE category. A few small listings landed in a different cluster than the rest of
# their model (for example a Kindle Paperwhite listing in the streaming cluster), so we use the majority category.
named_mask = reviews["product"].isin(PRODUCT_NAMES.values())
majority = reviews[named_mask].groupby("product")["cluster_name"].agg(lambda s: s.value_counts().index[0])
moved = (reviews.loc[named_mask, "cluster_name"] != reviews.loc[named_mask, "product"].map(majority)).sum()
reviews.loc[named_mask, "cluster_name"] = reviews.loc[named_mask, "product"].map(majority)
print("Reviews moved to their product's majority category:", moved)

print(len(reviews), "reviews |", reviews["cluster_name"].nunique(), "categories")
reviews["cluster_name"].value_counts()

FileNotFoundError: [Errno 2] No such file or directory: 'reviews_clustered.csv'

In [ ]:
# Check the names: raw names (most common first) vs verified name
check = (reviews.groupby("asin")
         .agg(reviews=("text", "size"),
              raw_names=("name_clean", lambda s: " | ".join(s.dropna().value_counts().head(2).index.str[:35])),
              verified=("product", "first"))
         .sort_values("reviews", ascending=False).head(18))
check

## 2. Statistics and complaint themes per product

**Ranking.** Listings of the same model are merged (for example the 16 GB and 32 GB Fire HD 8), and only products with a verified name and at least `MIN_REVIEWS_RANK` reviews are ranked. With `RANK_BY = "popularity"` (the default) the top 3 are the most-reviewed models, which are the ones readers are most likely to buy. With `RANK_BY = "rating"` the top 3 are the best-rated models, using a *smoothed* average that pulls products with few reviews toward the category average, so 5 reviews averaging 5.0 stars cannot beat 3,000 reviews averaging 4.7. In both cases the "worst" product is the one with the lowest smoothed rating among the ranked products.

**Complaints.** For each product we take its negative reviews (1-2 stars, widened to 1-3 stars if there are too few) and count how many mention each complaint theme. Themes are keyword patterns, so the results are easy to verify. Each percentage is the share of that product's negative reviews mentioning the theme.

In [ ]:
THEMES = [
    # (label, regex pattern, keyword used to check the article mentions it)
    ("battery life",           r"battery|drain|doesn.?t hold a charge|won.?t charge|dies quickly", "battery"),
    ("screen quality",         r"screen|display|resolution|glare|cracked", "screen"),
    ("slow performance",       r"\bslow|laggy|\blag\b|freez|sluggish|crash|glitch", "slow"),
    ("ads and special offers", r"\bads?\b|advertis|special offers|lock ?screen", "ad"),
    ("app availability",   r"google play|play store|\bapps?\b", "app"),
    ("voice recognition",      r"alexa|voice|understand|recogni|mishear", "voice"),
    ("Wi-Fi and connectivity", r"wi-?fi|connect|bluetooth|disconnect|internet", "wi-fi|connect"),
    ("storage and memory",     r"storage|memory|\bgb\b|sd card", "storage|memory"),
    ("sound quality",          r"sound|speaker|volume|audio|bass", "sound"),
    ("reliability",            r"broke|stopped working|defective|quit working|died|replacement", "reliab|broke|stopped|defect"),
    ("customer service",       r"customer service|support|warranty|refund", "service|support"),
    ("price and value",        r"price|expensive|overpriced|waste of money|not worth", "price|value"),
    ("setup and ease of use",  r"setup|set up|confusing|difficult|hard to|instructions|complicated", "setup|ease|difficult"),
    ("missing features",       r"doesn.?t have|does not have|can.?t (?:use|install)|wish it|no (?:google|camera)", "missing|feature"),
    ("plug and cable fit",     r"prong|plug|outlet|\bcord\b|cable|too short", "plug|cable|cord"),
]
THEME_PAT = {label: pat for label, pat, _ in THEMES}
THEME_KEY = {label: key for label, _, key in THEMES}

stats = (reviews.groupby(["cluster_name", "product"])
         .agg(n=("rating", "size"), avg=("rating", "mean"),
              pct_neg=("rating", lambda s: (s <= 2).mean() * 100),
              pct_pos=("rating", lambda s: (s >= 4).mean() * 100))
         .reset_index())
stats.sort_values(["cluster_name", "n"], ascending=[True, False]).round(1).head(20)

In [ ]:
POSITIVE_THEMES = [
    ("easy to use",                  r"easy to (?:use|set up|navigate)|easy setup|user.?friendly|simple to use"),
    ("price and value",              r"price|cheap|affordable|inexpensive|bargain|worth|great deal"),
    ("screen and display",           r"screen|display|\bclear\b|crisp|bright"),
    ("battery life",                 r"battery"),
    ("sound quality",                r"sound|speaker|volume|audio|bass"),
    ("reading experience",           r"\bread(?:ing|er|ers)?\b|books?\b|glare|backlight|e-?ink"),
    ("kids and family",              r"\bkids?\b|child|grandson|granddaughter|grandchild|daughter|parental"),
    ("voice control and smart home", r"alexa|voice|smart home|lights|music|weather|timer"),
    ("speed and performance",        r"\bfast\b|quick|smooth|responsive|speed"),
    ("streaming and video",          r"stream|netflix|movies?|\bvideos?\b|prime|roku|\b4k\b"),
    ("portability and size",         r"portable|portability|lightweight|compact|travel|\bsmall\b"),
    ("design and build",             r"design|sleek|sturdy|durable|build quality"),
    ("fast charging",                r"fast charg|charges? (?:fast|quickly)|quick charg"),
]

def praise_themes(g, top=3, min_share=0.10, min_count=5):
    # what reviewers like: share of the product's 4-5 star reviews that mention each theme
    low = g[g["rating"] >= 4]["text"].str.lower()
    res = []
    for label, pat in POSITIVE_THEMES:
        hits = low.str.contains(pat, regex=True)
        if hits.sum() >= min_count:
            res.append((label, hits.mean()))
    res = sorted(res, key=lambda x: -x[1])[:top]
    return [(l, int(round(s * 100))) for l, s in res if s >= min_share]

def negative_pool(g):
    neg = g[g["rating"] <= 2]
    cut = "1-2"
    if len(neg) < MIN_NEG:
        neg, cut = g[g["rating"] <= 3], "1-3"
    return neg, cut

def complaint_themes(neg, top=3, min_share=0.06, min_count=2):
    # a theme needs at least min_count matching reviews, so one stray review is never a "top complaint"
    low = neg["text"].str.lower()
    res = []
    for label, pat, _ in THEMES:
        hits = low.str.contains(pat, regex=True)
        if hits.sum() >= min_count:
            res.append((label, hits.mean()))
    res = sorted(res, key=lambda x: -x[1])
    return [(l, s) for l, s in res[:top] if s >= min_share]

def example_complaints(neg, label, n=2):
    low = neg["text"].str.lower()
    positive = low.str.contains(r"excellent|love it|loves it|perfect|awesome|amazing|fantastic|highly recommend", regex=True)
    mask = low.str.contains(THEME_PAT[label], regex=True) & ~positive & neg["text"].str.len().between(60, 220)
    cand = neg[mask]
    return cand.sample(min(n, len(cand)), random_state=SEED)["text"].tolist()

def distinctive_terms(cat_df, top=5, eps=0.005):
    # words that reviewers use much more for this product than for the other products in the category
    out = {}
    counts = cat_df["product"].value_counts()
    prods = counts[counts >= MIN_REVIEWS_RANK].index
    if len(prods) < 2:
        return out
    cv = CountVectorizer(binary=True, stop_words=STOP, min_df=10, token_pattern=r"[a-zA-Z]{3,}")
    try:
        X = cv.fit_transform(cat_df["text"])
    except ValueError:
        return out
    terms = np.array(cv.get_feature_names_out())
    for p in prods:
        m = (cat_df["product"] == p).values
        p_in = np.asarray(X[m].mean(axis=0)).ravel()
        p_out = np.asarray(X[~m].mean(axis=0)).ravel()
        score = p_in * np.log((p_in + eps) / (p_out + eps))
        ranked_terms = [t for t in terms[np.argsort(score)[::-1][:40]] if t not in GENERIC]
        out[p] = ranked_terms[:top]
    return out

## 3. The fact sheet for each category

For every category we pick the top 3 products and the worst product, and collect their numbers, distinguishing words, complaint themes and example complaints. This is the only information the language model will see.

If a category has fewer than two products with enough reviews, we rank the biggest products anyway and add a note saying the data is limited, so the article stays cautious. A category with fewer than `MIN_CATEGORY_REVIEWS` reviews (for example a tiny group of 34 phone cases) is skipped entirely: any article about it would be built on a handful of reviews.

In [ ]:
def product_entry(name, cdf, st_row, terms):
    g = cdf[cdf["product"] == name]
    neg, cut = negative_pool(g)
    themes = complaint_themes(neg)
    quotes = example_complaints(neg, themes[0][0]) if themes else []
    return {"name": name, "reviews": int(st_row["n"]), "avg": round(float(st_row["avg"]), 1),
            "pct_neg": int(round(st_row["pct_neg"])), "pct_pos": int(round(st_row["pct_pos"])),
            "mentions": terms.get(name, []), "neg_cut": cut, "n_neg": int(len(neg)),
            "complaints": [(l, int(round(s * 100))) for l, s in themes], "quotes": quotes,
            "praise": praise_themes(g)}

def build_facts(cat):
    cdf = reviews[reviews["cluster_name"] == cat]
    st = stats[stats["cluster_name"] == cat].copy()
    cat_avg = cdf["rating"].mean()
    st["score"] = (st["n"] * st["avg"] + SMOOTHING * cat_avg) / (st["n"] + SMOOTHING)

    # only products with a verified name can be ranked (never "Other product B00...")
    named = st[~st["product"].str.startswith("Other product") & ~st["product"].str.contains("model unclear")]
    pool = named if len(named) else st
    qualified = pool[pool["n"] >= MIN_REVIEWS_RANK]
    limited = len(qualified) == 0
    if limited:                                   # nothing has enough reviews: take the biggest ones, flagged as tentative
        qualified = pool.sort_values("n", ascending=False).head(3)
    single = len(qualified) == 1                  # only one product to write about
    sort_col = "n" if RANK_BY == "popularity" else "score"
    ranked = qualified.sort_values(sort_col, ascending=False)

    terms = distinctive_terms(cdf)
    top = [product_entry(r["product"], cdf, r, terms) for _, r in ranked.head(3).iterrows()]
    w = qualified.sort_values("score").iloc[0]          # lowest smoothed rating among all ranked products
    worst = product_entry(w["product"], cdf, w, terms)
    return {"category": cat, "n_reviews": int(len(cdf)), "n_products": int(st.shape[0]),
            "avg": round(float(cat_avg), 1), "pct_neg": int(round((cdf["rating"] <= 2).mean() * 100)),
            "limited": limited, "single": single, "top": top, "worst": worst,
            "worst_is_also_top": worst["name"] in [t["name"] for t in top]}

def fmt_product(p, rank=None):
    head = f"{rank}. " if rank else ""
    lines = [f"{head}{p['name']}: {p['reviews']} reviews, average {p['avg']} out of 5, "
             f"{p['pct_pos']}% positive (4-5 stars), {p['pct_neg']}% negative (1-2 stars)."]
    if p["mentions"]:
        lines.append(f"   Topics reviewers mention more often for this product than for the others here "
                     f"(topics only, not praise or complaints): {', '.join(p['mentions'])}.")
    if p["praise"]:
        pr = "; ".join(f"{l} ({s}%)" for l, s in p["praise"])
        lines.append(f"   What reviewers praise (share of its 4-5 star reviews): {pr}.")
    else:
        lines.append("   No praise theme stands out in its positive reviews.")
    if p["complaints"]:
        comp = "; ".join(f"{l} ({s}%)" for l, s in p["complaints"])
        lines.append(f"   Top complaints, as a share of its {p['neg_cut']}-star reviews ({p['n_neg']} reviews): {comp}.")
        for q in p["quotes"]:
            lines.append(f'   Example complaint: "{q}"')
    else:
        lines.append("   No complaint theme stands out in its negative reviews.")
    return "\n".join(lines)

def facts_to_text(f):
    out = [f"CATEGORY: {f['category']}",
           f"Reviews in category: {f['n_reviews']} across {f['n_products']} products; "
           f"average rating {f['avg']} out of 5; {f['pct_neg']}% negative reviews."]
    if f["limited"]:
        out.append("NOTE: few products have many reviews here, so all conclusions are tentative.")
    if f["single"]:
        out.append("\nTHE ONE PRODUCT WITH ENOUGH REVIEWS TO ANALYSE:")
        out.append(fmt_product(f["top"][0]))
        out.append("NOTE: only this product has enough reviews to analyse. Write about it alone, do not compare it "
                   "with other products, and do not mention any other products.")
        if f["top"][0]["avg"] >= 4.0:
            out.append("NOTE: it is rated well overall, so describe its weaknesses as relatively minor.")
        return "\n".join(out)
    order = "the most-reviewed first" if RANK_BY == "popularity" else "best rated first"
    out.append(f"\nTOP PRODUCTS ({order}):")
    out += [fmt_product(p, i) for i, p in enumerate(f["top"], 1)]
    label = "LOWEST-RATED PRODUCT" if f["worst_is_also_top"] else "WORST PRODUCT"
    out.append(f"\n{label}:")
    out.append(fmt_product(f["worst"]))
    if f["worst"]["avg"] >= 4.0:
        out.append("NOTE: even this product is rated well overall, so describe its weaknesses as relatively minor.")
    return "\n".join(out)

categories = list(reviews["cluster_name"].value_counts().index)
facts = {c: build_facts(c) for c in categories}
fact_text = {c: facts_to_text(facts[c]) for c in categories}
print(fact_text[categories[0]])

Read a fact sheet critically. Do the top products and complaints look right for that category? This is the ground truth that the articles will be checked against. If something looks wrong, fix it here (for example in `PRODUCT_NAMES` or `THEMES`), not in the prompt.

## 4. Load the language model

I recommend **`Qwen/Qwen2.5-3B-Instruct`**: it follows instructions well, fits a free Colab T4 in half precision (roughly 7 GB of GPU memory), and needs no access request. Alternatives:

- `Qwen/Qwen2.5-1.5B-Instruct`: use this on a CPU or small GPU. It is faster, but its writing is weaker.
- `meta-llama/Llama-3.2-3B-Instruct`: comparable quality, but you must accept the license on Hugging Face and log in with a token.
- A 7B model (for example `Qwen/Qwen2.5-7B-Instruct`) needs 4-bit loading with `bitsandbytes` to fit a T4, and is slower.

The brief asks for 1B-8B parameters and says not to go larger.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"     # use "Qwen/Qwen2.5-1.5B-Instruct" without a GPU

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dtype = torch.float16 if device == "cuda" else torch.float32
kwargs = {"device_map": "auto"} if device == "cuda" else {}
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype, **kwargs)
except TypeError:                                   # older transformers versions call it torch_dtype
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, **kwargs)
model.eval()

In [ ]:
def generate(system, user):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    set_seed(SEED)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                             temperature=0.2, top_p=0.9, repetition_penalty=1.05,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

## 5. Three prompt variants, scored automatically

| Variant | Idea |
|---|---|
| **A: basic** | One plain instruction plus the facts. The "does it work at all" baseline. |
| **B: structured** | A system message with strict rules (use only the facts, copy names and numbers exactly) and a required article structure. |
| **C: structured + style** | B, plus a style guide (direct comparisons, honest and concrete, no hype), similar to buying-guide sites such as Wirecutter. |

Low temperature (0.2) keeps the model close to the facts.

In [ ]:
SYSTEM_A = "You are a helpful assistant."

SYSTEM_B = (
    "You are an editor at a consumer-electronics buying-guide website. You write honest, concrete "
    "articles based ONLY on the review data you are given.\n"
    "Rules:\n"
    "- Use only facts that appear in the data. Never mention a product, model, feature, price or specification "
    "that is not in the data, and never assume what a product 'also' offers.\n"
    "- Praise a product only for what appears under 'What reviewers praise'. Describe problems only with the listed complaints.\n"
    "- Copy product names exactly as written. Quote numbers (ratings, percentages, review counts) exactly as given.\n"
    "- Each percentage belongs to the complaint or praise it is listed next to. Never merge two items under one percentage.\n"
    "- 'Topics reviewers mention' are only topics, not praise and not complaints.\n"
    "- Example complaints are single reviews. If you quote one, say 'one reviewer wrote'. Never present it as a general trend.\n"
    "- If the data says conclusions are tentative, say so in the article."
)

TOP_HEADING = "The most popular models" if RANK_BY == "popularity" else "Our top picks"
OUTLINE_INTRO = "Use exactly this outline (markdown). The text in [brackets] is an instruction: never copy it.\n"

def outline(cat):
    f = facts[cat]
    if f["single"]:
        return (OUTLINE_INTRO +
                "# [a specific title you write for this article]\n"
                "[two-sentence introduction, no heading]\n"
                "## What reviewers like\n[one short paragraph, using only 'What reviewers praise']\n"
                "## Weak spots\n[its main complaints, using only the listed complaints]\n"
                "## Bottom line\n[two sentences]\n")
    if f["worst"]["avg"] >= 4.0:
        weak_head = "The weakest of the group"
        weak_text = ("the lowest-rated product and its relative weakness, using only its listed complaints. "
                     "Say plainly that none of these products is a poor choice and that the differences are small")
    else:
        weak_head = "The one to skip"
        weak_text = "the lowest-rated product and why, using only its listed complaints"
    return (OUTLINE_INTRO +
            "# [a specific title you write for this article]\n"
            "[two-sentence introduction, no heading]\n"
            f"## {TOP_HEADING}\n[one short paragraph per product: what sets it apart, what reviewers praise, its main complaints]\n"
            f"## {weak_head}\n[{weak_text}]\n"
            "## Bottom line\n[two sentences]\n")

def user_prompt(variant, cat, facts_str):
    if variant == "A":
        return f"Write a short blog post about the product category '{cat}' based on these customer review facts:\n\n{facts_str}"
    style = ""
    if variant == "C":
        compare = "" if facts[cat]["single"] else "Compare the products with each other (for example 'unlike X, Y ...'). "
        style = f"Style: direct and practical, like a buying guide. {compare}Short sentences, no hype, no filler.\n"
    return (f"Write a blog-style article of 300-400 words for the category '{cat}'.\n{outline(cat)}{style}\n"
            f"DATA:\n{facts_str}")

VARIANTS = {"A": SYSTEM_A, "B": SYSTEM_B, "C": SYSTEM_B}

### Automatic checks

For every generated article we compute six scores between 0 and 1:

- **numbers_real:** the share of numbers in the article that also appear in the fact sheet (low means invented numbers)
- **products_covered:** the share of the ranked products (top 3 and worst) named in the article
- **complaints_covered:** the share of the listed complaint themes that the article mentions
- **length_ok:** 1 if the article has 250-500 words, lower the further it is outside
- **structure_ok:** has at least three headings, a section about the weakest product, and is not cut off mid-sentence
- **names_ok:** 1 if the article names no product-like phrase ("Echo Dot Plus", "Alexa Guard") that is missing from the fact sheet, 0 otherwise. The offending phrases are listed in the `unsupported` column.

The scores are simple and have limits. They cannot tell whether the writing is good or the claims are placed correctly (the model could attach a complaint to the wrong product). So also read a few articles yourself, below.

In [ ]:
def tidy(article):
    # remove "A Catchy Title:" or "A Specific Title:" if the model copies the instruction into its heading
    return re.sub(r"(?im)^(#+\s*)(?:a\s+)?(?:catchy|specific)\s+title\s*[:\-–]\s*", r"\1", article)

COMMON = {"is", "are", "the", "and", "with", "has", "have", "offers", "also", "but", "for", "remains", "stands",
          "excels", "boasts", "as", "in", "on", "to", "of", "that", "this", "these", "it", "its", "can", "will",
          "may", "does", "do", "was", "were", "a", "an", "or", "from", "by", "than", "which", "while", "our"}

def unsupported_names(article, facts_str):
    # product-like phrases ("Echo Dot Plus", "Alexa Guard") that never appear in the fact sheet
    body = "\n".join(article.split("\n")[1:])                 # skip the title line
    facts_low = facts_str.lower()
    bad = []
    for ph in re.findall(r"\b(?:Echo|Fire|Kindle|Alexa|Amazon)\b(?:\s+[A-Z][\w+\-]*){1,3}", body):
        words = ph.split()
        while len(words) > 1 and words[-1].lower() in COMMON:
            words.pop()
        # flagged only if some word of the phrase never appears in the fact sheet ("Plus", "Guard")
        missing = [w for w in words if not re.search(r"\b" + re.escape(w.lower().rstrip("s")), facts_low)]
        if missing:
            bad.append(" ".join(words))
    return bad

def score_article(article, f, facts_str):
    words = len(article.split())
    allowed = set(re.findall(r"\d+(?:\.\d+)?", facts_str)) | {"1", "2", "3", "4", "5"}
    nums = re.findall(r"\d+(?:\.\d+)?", article)
    numbers_real = np.mean([n in allowed for n in nums]) if nums else 1.0

    low = article.lower()
    products = [p["name"] for p in f["top"]] + [f["worst"]["name"]]
    products_covered = np.mean([p.lower() in low for p in set(products)])

    themes = {l for p in f["top"] + [f["worst"]] for l, _ in p["complaints"]}
    complaints_covered = (np.mean([bool(re.search(THEME_KEY[t], low)) for t in themes]) if themes else 1.0)

    length_ok = 1.0 if 250 <= words <= 500 else max(0.0, 1 - min(abs(words - 250), abs(words - 500)) / 250)
    headings = len(re.findall(r"^#+ ", article, flags=re.M))
    complete = article.rstrip()[-1:] in ".!?)\"'*"              # not cut off mid-sentence
    structure_ok = float(headings >= 3 and complete and bool(re.search(r"skip|avoid|worst|weakest|lowest|weak spot|drawback", low)))

    bad = unsupported_names(article, facts_str)
    return {"numbers_real": numbers_real, "products_covered": products_covered,
            "complaints_covered": complaints_covered, "length_ok": length_ok,
            "structure_ok": structure_ok, "names_ok": float(len(bad) == 0),
            "unsupported": ", ".join(bad[:3]), "words": words}

In [ ]:
import time

N_EVAL_CATEGORIES = None        # quick test first (3 articles); set to None for all categories

small = [c for c in categories if facts[c]["n_reviews"] < MIN_CATEGORY_REVIEWS]
if small:
    print("Skipping categories with too few reviews for a reliable article:", small)
eval_cats = [c for c in categories if c not in small]
if N_EVAL_CATEGORIES:
    eval_cats = eval_cats[:N_EVAL_CATEGORIES]

# articles already generated are kept, so re-running this cell resumes where it stopped
# (uncomment the next line to throw everything away and regenerate, e.g. after changing a prompt)
# articles = {}
if "articles" not in globals():
    articles = {}

total = len(eval_cats) * len(VARIANTS)
count, t_start = 0, time.time()
for cat in eval_cats:
    for v, system in VARIANTS.items():
        count += 1
        if (v, cat) in articles:
            continue
        t = time.time()
        articles[(v, cat)] = tidy(generate(system, user_prompt(v, cat, fact_text[cat])))
        print(f"[{count}/{total}] {cat} | prompt {v} | {time.time() - t:.0f}s | "
              f"{len(articles[(v, cat)].split())} words")
print(f"Total time: {(time.time() - t_start) / 60:.1f} min")

rows = [{"variant": v, "category": cat,
         **score_article(articles[(v, cat)], facts[cat], fact_text[cat])}
        for cat in eval_cats for v in VARIANTS]
evaluation = pd.DataFrame(rows)
score_cols = ["numbers_real", "products_covered", "complaints_covered", "length_ok", "structure_ok", "names_ok"]
evaluation["overall"] = evaluation[score_cols].mean(axis=1)
summary = evaluation.groupby("variant")[score_cols + ["overall", "words"]].mean().round(2)
summary

In [ ]:
evaluation.round(2).sort_values(["category", "variant"])

**How to read the results:** compare the variants on the columns, not only on `overall`. A variant with a high `overall` but low `numbers_real` is inventing numbers, which is worse than a missing paragraph. The variant with the best balance goes forward, and you should read at least one article from each variant to see *why* it scored as it did.

If no variant is good enough, there are three levers: improve the fact sheet (usually the most effective), try a larger model (up to 7B with 4-bit loading), or fine-tune a small model on examples. The brief suggests fine-tuning only if prompt engineering falls short.

## 6. Final articles

We use the variant with the best overall score. Because every article was already generated during the comparison, we reuse them.

In [ ]:
best = summary["overall"].idxmax()
print("Best prompt variant:", best)

final = {cat: articles[(best, cat)] for cat in eval_cats}
for cat in eval_cats:
    display(Markdown(final[cat] + "\n\n---"))

In [ ]:
with open("category_articles.md", "w", encoding="utf-8") as f:
    for cat in eval_cats:
        f.write(final[cat] + "\n\n---\n\n")

evaluation.to_csv("prompt_evaluation.csv", index=False)
with open("category_facts.json", "w", encoding="utf-8") as f:
    json.dump({c: facts[c] for c in eval_cats}, f, indent=2)
print("Saved category_articles.md, prompt_evaluation.csv and category_facts.json")

## Summary and limitations

| Part | Choice | Why |
|---|---|---|
| Facts | Computed with pandas, not by the model | Exact numbers; a small model would invent them |
| Product names | Verified table by product ID | The raw `name` column is unreliable |
| Ranking | Most-reviewed models (or best smoothed rating), minimum 30 reviews, same-model listings merged | Relevant picks; avoids ranking products with a handful of reviews |
| Complaints | Keyword themes over negative reviews | Easy to check and explain |
| Model | `Qwen2.5-3B-Instruct`, low temperature | Fits a free GPU, follows instructions |
| Evaluation | Three prompts, five automatic checks | Compare prompt variants objectively |

**Limitations to mention in the write-up:**
- The product names are my interpretation of noisy data, and some entries are marked "model unclear".
- Complaint themes are keyword matches: a review saying "no problems with the battery" would still count as a battery complaint.
- The star rating decides what counts as negative. You could instead use the sentiment model from Task 1 (the share of reviews it classifies as negative) as a second signal.
- Categories with few reviews, like accessories, give weak conclusions, and the article should say so.
- The automatic scores cannot catch a complaint attached to the wrong product, so read the final articles before relying on them.